Importing all necessary libraries

In [129]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from xgboost import XGBRegressor

from sklearn.model_selection import train_test_split, TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score, median_absolute_error

Importing features.parquet file

In [130]:
features_df = pd.read_parquet("data/processed/features.parquet")

In [131]:
features_df.head(5)

,flight_id,taxi_start,taxi_end,taxi_time_s,typecode,queue_size,avg_taxi_out_time_6h,avg_taxi_out_time_1h,avg_taxi_out_time_15min,n_intersections_passed,sum_seg_length_m,sum_p10_hist_seg_time,sum_median_hist_seg_time,sum_p90_hist_seg_time,TO_runway_28,TO_runway_32,TO_runway_16,TO_runway_10,TO_runway_34,pushback,month,day_of_week,minute_of_day,month_sin,month_cos,day_of_week_sin,day_of_week_cos,minute_of_day_sin,minute_of_day_cos,temp_at_taxi_start,spread_at_taxi_start
0,GSW6KP_4832,2024-12-01 04:44:54+00:00,2024-12-01 04:58:20+00:00,806.0,A320,0,NaN,NaN,NaN,4,711.4,NaN,NaN,NaN,0,1,0,0,0,1,12,6,284,-2.449294e-16,1.0,-0.781831,0.62349,0.945519,0.325568,0.3,2.4
1,EDW200E_4558,2024-12-01 04:58:35+00:00,2024-12-01 05:07:58+00:00,563.0,A320,0,806.000000,806.000000,806.0,7,1190.3,NaN,NaN,NaN,0,1,0,0,0,1,12,6,298,-2.449294e-16,1.0,-0.781831,0.62349,0.963630,0.267238,0.3,2.4
2,EDW120_4456,2024-12-01 05:18:50+00:00,2024-12-01 05:25:38+00:00,408.0,A320,0,684.500000,684.500000,563.0,4,740.8,NaN,NaN,NaN,0,1,0,0,0,1,12,6,318,-2.449294e-16,1.0,-0.781831,0.62349,0.983255,0.182236,0.2,2.4
3,EDW15T_4539,2024-12-01 05:31:07+00:00,2024-12-01 05:35:22+00:00,255.0,A320,0,592.333333,592.333333,408.0,7,1092.5,NaN,NaN,NaN,0,1,0,0,0,1,12,6,331,-2.449294e-16,1.0,-0.781831,0.62349,0.992005,0.126199,0.3,2.3
4,EDW212R_4430,2024-12-01 05:34:06+00:00,2024-12-01 05:46:14+00:00,728.0,A320,1,592.333333,592.333333,408.0,11,1911.9,NaN,NaN,NaN,0,1,0,0,0,1,12,6,334,-2.449294e-16,1.0,-0.781831,0.62349,0.993572,0.113203,0.3,2.3


Sorting the DataFrame based on Time

In [132]:
features_df = (
    features_df
    .sort_values("taxi_start")
    .reset_index(drop=True)
)

Defining X (Features)

In [ ]:
X = features_df[["typecode",
                 "queue_size",
                 "avg_taxi_out_time_6h",
                 "avg_taxi_out_time_1h",
                 "avg_taxi_out_time_15min",
                 "n_intersections_passed",
                 "sum_seg_length_m",
                 "sum_p10_hist_seg_time",
                 "sum_median_hist_seg_time",
                 "sum_p90_hist_seg_time",
                 "pushback",
                 "TO_runway_28",
                 "TO_runway_32",
                 "TO_runway_16",
                 "TO_runway_10",
                 "TO_runway_34",
                 "month_sin",
                 "month_cos",
                 "day_of_week_sin",
                 "day_of_week_cos",
                 "minute_of_day_sin",
                 "minute_of_day_cos",
                 "temp_at_taxi_start",
                 "spread_at_taxi_start"
]]

Defining Y (Target Variable)

In [134]:
y = features_df[["taxi_time_s"]]

The following code implements a rolling-origin time series cross-validation procedure using expanding training windows. For each fold, the training data consists of all flights prior to the validation period, while the validation set contains flights from a future unseen month. This ensures that the model is always trained on past data and evaluated on future data, thereby avoiding temporal data leakage.

Within each fold, the feature matrix (`X_train`, `X_val`) and target variable (`y_train`, `y_val`) are created using boolean masks based on the `taxi_start` timestamp. An XGBoost regressor is then trained on the training subset and evaluated on the validation subset using the Mean Absolute Error (MAE) Root Mean Squared Error (RMSE), R Squared Score (R^2) and Median Absolute Error (MedAE). The process is repeated for all folds to obtain a robust estimate of model performance across different operational periods.


In [135]:
folds = [
    ("2025-03-01", "2025-04-01"),
    ("2025-05-01", "2025-06-01"),
    ("2025-07-01", "2025-08-01"),
    ("2025-09-01", "2025-10-01"),
    ("2025-11-01", "2025-12-01"),
]

results = []

for fold, (val_start, val_end) in enumerate(folds, start=1):

    print(f"\n========== FOLD {fold} ==========")

    train_mask = features_df["taxi_start"] < val_start

    val_mask = (
        (features_df["taxi_start"] >= val_start) &
        (features_df["taxi_start"] < val_end)
    )

    X_train = features_df.loc[train_mask, X.columns]
    y_train = features_df.loc[train_mask, "taxi_time_s"]

    X_val = features_df.loc[val_mask, X.columns]
    y_val = features_df.loc[val_mask, "taxi_time_s"]

    print(f"Training samples:   {len(X_train):,}")
    print(f"Validation samples: {len(X_val):,}")

    model = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        random_state=42,
        enable_categorical=True,
        objective='reg:squarederror',
    )

    model.fit(X_train, y_train)

    model.fit(X_train, y_train)

    importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
    })

    importance_df = importance_df.sort_values(
    "importance",
    ascending=False
    )

    print(importance_df)

    y_pred = model.predict(X_val)    

    y_pred = model.predict(X_val)

    mae = mean_absolute_error(y_val, y_pred)
    rmse = root_mean_squared_error(y_val, y_pred)
    medae = median_absolute_error(y_val, y_pred)
    r_2 = r2_score(y_val,y_pred)

    results.append({
    "fold": fold,
    "mae": mae,
    "rmse": rmse,
    "medae": medae,
    "r2": r_2
})

    print(f"MAE:  {mae:.2f} s")
    print(f"RMSE: {rmse:.2f} s")
    print(f"MedAE: {medae:.2f} s")
    print(f"R^2: {r_2:.2f}")

    
results_df = pd.DataFrame(results)

results_df



========== FOLD 1 ==========
Training samples:   11,889
Validation samples: 4,271
                     feature  importance
9      sum_p90_hist_seg_time    0.386156
8   sum_median_hist_seg_time    0.060653
7      sum_p10_hist_seg_time    0.055773
11              TO_runway_28    0.051787
12              TO_runway_32    0.050622
10                  pushback    0.037403
5     n_intersections_passed    0.035185
18        temp_at_taxi_start    0.033003
1                 queue_size    0.032008
0                   typecode    0.030090
17         minute_of_day_cos    0.027956
3       avg_taxi_out_time_1h    0.027885
2       avg_taxi_out_time_6h    0.025972
16         minute_of_day_sin    0.025821
19      spread_at_taxi_start    0.023746
6           sum_seg_length_m    0.022778
14              TO_runway_10    0.022518
15              TO_runway_34    0.021033
4    avg_taxi_out_time_15min    0.017362
13              TO_runway_16    0.012250
MAE:  188.08 s
RMSE: 303.28 s
MedAE: 127.54 s
R^2: 0.30


,fold,mae,rmse,medae,r2
0,1,188.077565,303.279520,127.539368,0.303873
1,2,184.804685,306.099288,121.196808,0.307220
2,3,176.027039,279.399398,124.024414,0.356056
3,4,164.791374,267.069192,112.842651,0.381061
4,5,176.659275,268.738007,123.752197,0.533335
